In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version5.data import trainLoader
from source.version5.model import EfficientModel
from source.version5.train import trainModel
from source.version5.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/cropped/train/train/'
    loader['label_path'] = '../../data/cropped/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 7
    params['num_workers'] = 4
    params['drop_last'] = True
    sampler = BalanceClassSampler(labels = train.labels(), mode="downsampling")
    train = DataLoader(train, shuffle=True, **params)
    valid = DataLoader(valid, **params)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=3e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.5, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version5/model_{}.pt'.format(fold)
    trainer['epochs'] = 15
    trainer['batch'] = 7
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 35557 Valid Images: 6478
Loaded pretrained weights for efficientnet-b5


100% 35553/35553 [34:41<00:00, 17.08it/s, trn_ls=0.4915, val_ls=0.1876, val_mt=0.9023]
100% 35553/35553 [34:20<00:00, 17.25it/s, trn_ls=0.3415, val_ls=0.1775, val_mt=0.9077]
100% 35553/35553 [34:25<00:00, 17.21it/s, trn_ls=0.2950, val_ls=0.1708, val_mt=0.9162]
100% 35553/35553 [34:47<00:00, 17.03it/s, trn_ls=0.2619, val_ls=0.1585, val_mt=0.9305]
100% 35553/35553 [34:46<00:00, 17.04it/s, trn_ls=0.2425, val_ls=0.1582, val_mt=0.9313]
100% 35553/35553 [35:15<00:00, 16.81it/s, trn_ls=0.2235, val_ls=0.1632, val_mt=0.9281]
 31% 11116/35553 [10:30<25:07, 16.21it/s, trn_ls=0.20880]

In [ ]:
train(1)

Train Images: 35416 Valid Images: 6474
Loaded pretrained weights for efficientnet-b5


In [ ]:
train(2)

Train Images: 35523 Valid Images: 6428
Loaded pretrained weights for efficientnet-b5


In [ ]:
train(3)

Train Images: 35292 Valid Images: 6678
Loaded pretrained weights for efficientnet-b5


 37% 13076/35287 [13:07<23:18, 15.88it/s, trn_ls=0.63140]

In [ ]:
train(4)